# Task 1

## 1.a. Regression uchun Analitik Yechim

Linear Regression uchun analitik yechim Mean Squared Error (MSE) ni minimallashtirib topiladi:

$$\min_{\theta} L(\theta) = \min_{\theta} \|y - X\theta\|_2^2$$

Hosilani olib, nolga tenglab:

$$\frac{\partial L}{\partial \theta} = -2X^T(y - X\theta) = 0$$

**Analitik Yechim (Vektor ko'rinishi):**

$$\boxed{\theta = (X^T X)^{-1} X^T y}$$

Bu yerda:
- $X$ — $(n \times d)$ o'lchamli design matrix
- $y$ — $(n \times 1)$ o'lchamli target vektor
- $\theta$ — modelning optimal parametrlari
- $(X^T X)^{-1}X^T$ — Moore-Penrose psevdoinvers

---

## 1.b. Regularization bilan Loss Function

Regularization bilan umumiy loss function:

$$\min_{\theta} \sum_{i=1}^{N} L(f(x_i, \theta), y_i) + \lambda R(\theta)$$

Bu yerda:
- $L$ — loss function (masalan, MSE)
- $R(\theta)$ — regularization haddi
- $\lambda$ — regularization kuchi (hyperparameter)

---

## 1.c. Feature Selection uchun nima uchun L1 Regularization

L1 regularization loss function ga $R(\theta) = \|\theta\|_1 = \sum_{i=1}^{d} |\theta_i|$ jarima haddi qo'shadi:

$$L_{L1} = \|y - X\theta\|_2^2 + \lambda \sum_{i=1}^{d} |\theta_i|$$

**Asosiy xususiyat**: L1 regularization ko'p koeffitsientlar aynan nolga teng bo'ladigan **sparse yechimlar** hosil qiladi. Buning sababi:
1. L1 norma olmos shaklida cheklash sohasiga ega
2. Optimal yechim ko'pincha ushbu olmosning burchaklarida joylashadi
3. Ushbu burchaklar nol koeffitsientlarga mos keladi

**Feature Selection uchun afzalligi**: Zaif feature'lar avtomatik ravishda nol koeffitsient oladi, bu ularni maxsus feature engineering siz modeldan samarali olib tashlaydi.

---

## 1.d. Linear Modellar bilan Chiziqli Bo'lmagan Bog'liqliklarni Moslashtirish

Linear Regression faqat chiziqli funksiyalarga moslashsa ham, chiziqli bo'lmagan munosabatlarni quyidagilar orqali modellashtirish mumkin:

1. **Polynomial Features**: Feature'larni $x \mapsto [x, x^2, x^3, \ldots]$ ga o'zgartirish
2. **Chiziqli Bo'lmagan Transformatsiyalar**: $\log(x)$, $\sqrt{x}$, $\sin(x)$, $e^x$ kabi funksiyalardan foydalanish
3. **Kernel Methods** (ilg'or): Yuqori o'lchamli fazoiga tasvirlash

Misol: $y = a \cdot \log(x) + b$ ni moslash uchun transformatsiya qilingan feature $x' = \log(x)$ ni linear regression bilan ishlating.

Model o'zi transformatsiya qilingan fazoda chiziqli bo'lib qoladi, bu esa uni transformatsiya qilingan feature'larda linear model sifatida ko'rsatadi.

In [56]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# data pakpa ml1 digi bilan bir xil ushattan osa bo'ladi
df = pd.read_json("data/train.json")
df2 = pd.read_json("data/test.json")

# Part 2: Intro Data Analysis

In [58]:
df['features'] = df['features'].apply(lambda x: [f.strip().replace('[','').replace(']','').replace("'",'').replace('"', '').replace(' ', '') for f in x])
df2['features'] = df2['features'].apply(lambda x: [f.strip().replace('[','').replace(']','').replace("'",'').replace('"', '').replace(' ', '') for f in x])

all_features = []
for index, row in df.iterrows():
    for item in row['features']:
        all_features.append(item)

cnt = Counter(all_features)
top_features = [f for f, c in cnt.most_common(20) if f != '']
top_features

['Elevator',
 'CatsAllowed',
 'HardwoodFloors',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace']

In [59]:
for feature in top_features:
    df[feature] = df['features'].apply(lambda x: 1 if feature in x else 0)
    df2[feature] = df2['features'].apply(lambda x: 1 if feature in x else 0)

feature_list = top_features + ['bathrooms', 'bedrooms']
feature_list

['Elevator',
 'CatsAllowed',
 'HardwoodFloors',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace',
 'bathrooms',
 'bedrooms']

### **Target Variable**

In [60]:
X = df[feature_list].values
y = df['price'].values
X_test = df2[feature_list].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=21)

print(f"Training set size: {X_train.shape}")
print(f"Validation set size: {X_val.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Features: {X_train.shape}")

Training set size: (39481, 22)
Validation set size: (9871, 22)
Test set size: (74659, 22)
Features: (39481, 22)


### **Baholash Metrikasi Yordamchi (Evaluation metrics helper) Funksiyalari**

In [61]:
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def r2(y_true, y_pred):
    return r2_score(y_true, y_pred)

### Part 4: **Models implementation — Linear regression**


In [62]:
# 1. Linear Regression — 3 ta implementatsiya

# Stochastic Gradient Descent
class LinearRegressionSGD:
    """
    Deterministik SGD: random_state orqali har doim bir xil natija beradi.
    Deterministik model — bir xil input bilan har doim bir xil output qaytaradi.
    Bu random_state yordamida np.random.seed() o'rnatish orqali ta'minlanadi.
    """
    def __init__(self, learning_rate=0.01, n_epochs=100, batch_size=32, random_state=21):
        self.learning_rate = learning_rate
        self.n_epochs      = n_epochs
        self.batch_size    = batch_size
        self.random_state  = random_state
        self.weights       = None
        self.bias          = None

    def fit(self, X, y):
        np.random.seed(self.random_state)          # deterministik qilish
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias    = 0.0

        for epoch in range(self.n_epochs):
            indices   = np.random.permutation(n_samples)
            X_shuffle = X[indices]
            y_shuffle = y[indices]

            for start in range(0, n_samples, self.batch_size):
                end     = min(start + self.batch_size, n_samples)
                X_batch = X_shuffle[start:end]
                y_batch = y_shuffle[start:end]

                y_pred = X_batch @ self.weights + self.bias
                error  = y_pred - y_batch

                dw = (1 / len(X_batch)) * X_batch.T @ error
                db = (1 / len(X_batch)) * np.sum(error)

                self.weights -= self.learning_rate * dw
                self.bias    -= self.learning_rate * db

    def predict(self, X):
        return X @ self.weights + self.bias


class LinearRegressionAnalytical:
    """
    Analitik yechim: θ = (X^T X)^{-1} X^T y
    Gradient descent ishlatilmaydi — to'g'ridan-to'g'ri hisob-kitob.
    """
    def __init__(self):
        self.theta = None

    def fit(self, X, y):
        # Bias uchun birlar ustuni qo'shamiz
        X_b = np.c_[np.ones(X.shape[0]), X]
        self.theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y

    def predict(self, X):
        X_b = np.c_[np.ones(X.shape[0]), X]
        return X_b @ self.theta

# Batch Gradient Descent
class LinearRegressionBatchGD:
    """
    Batch GD: har iteratsiyada BUTUN dataset bo'yicha gradient hisoblanadi.
    SGD dan farqi: mini-batch yo'q, barcha ma'lumot bir vaqtda ishlatiladi.
    Deterministik model — random_state orqali har doim bir xil natija beradi.
    """
    def __init__(self, learning_rate=0.01, n_epochs=100, random_state=21):
        self.learning_rate = learning_rate
        self.n_epochs      = n_epochs
        self.random_state  = random_state
        self.weights       = None
        self.bias          = None

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias    = 0.0

        for epoch in range(self.n_epochs):
            y_pred = X @ self.weights + self.bias
            error  = y_pred - y

            dw = (1 / n_samples) * X.T @ error
            db = (1 / n_samples) * np.sum(error)

            self.weights -= self.learning_rate * dw
            self.bias    -= self.learning_rate * db

    def predict(self, X):
        return X @ self.weights + self.bias

### **Linear Regression Implementation — Training and Comparison**


In [63]:
# Step 1: Define R² (Coefficient of Determination) function
def calculate_r2(y_true, y_pred):
    """
    R² (Coefficient of Determination) - quality metric for regression
    
    Formula: R² = 1 - (SS_res / SS_tot)
    Where:
        SS_res = Σ(y_true - y_pred)² - residual sum of squares
        SS_tot = Σ(y_true - mean(y_true))² - total sum of squares
    
    What is R²:
        - R² = 1: Perfect fit (model explains all variance)
        - R² = 0: Model is as good as predicting sample mean
        - R² < 0: Model is worse than predicting mean
    """
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)


In [ ]:
# Test R² function
y_test_dummy = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred_dummy = np.array([1.1, 2.2, 2.9, 4.1, 4.8])
r2_test = calculate_r2(y_test_dummy, y_pred_dummy)
print(f"R² function verification: {r2_test:.4f}")

R² function verification: 0.9890


In [65]:
# 2.1. Stochastic Gradient Descent
lr_sgd = LinearRegressionSGD(learning_rate=0.001, n_epochs=50, batch_size=32, random_state=21)
lr_sgd.fit(X_train, y_train)
y_pred_train_sgd = lr_sgd.predict(X_train)
y_pred_val_sgd = lr_sgd.predict(X_val)

mae_train_sgd = mae(y_train, y_pred_train_sgd)
mae_val_sgd = mae(y_val, y_pred_val_sgd)
rmse_train_sgd = rmse(y_train, y_pred_train_sgd)
rmse_val_sgd = rmse(y_val, y_pred_val_sgd)
r2_train_sgd = calculate_r2(y_train, y_pred_train_sgd)
r2_val_sgd = calculate_r2(y_val, y_pred_val_sgd)

print(f"1 SGD (Stochastic Gradient Descent - Deterministik)")
print(f"    MAE:  train={mae_train_sgd:.2f}, validation={mae_val_sgd:.2f}")
print(f"    RMSE: train={rmse_train_sgd:.2f}, validation={rmse_val_sgd:.2f}")
print(f"    R²:   train={r2_train_sgd:.4f}, validation={r2_val_sgd:.4f}")

1 SGD (Stochastic Gradient Descent - Deterministik)
    MAE:  train=1226.25, validation=1027.77
    RMSE: train=24567.49, validation=2185.59
    R²:   train=0.0055, validation=0.3310


In [66]:
# 2.2. Batch Gradient Descent
lr_bgd = LinearRegressionBatchGD(learning_rate=0.001, n_epochs=50, random_state=21)
lr_bgd.fit(X_train, y_train)
y_pred_train_bgd = lr_bgd.predict(X_train)
y_pred_val_bgd = lr_bgd.predict(X_val)

mae_train_bgd = mae(y_train, y_pred_train_bgd)
mae_val_bgd = mae(y_val, y_pred_val_bgd)
rmse_train_bgd = rmse(y_train, y_pred_train_bgd)
rmse_val_bgd = rmse(y_val, y_pred_val_bgd)
r2_train_bgd = calculate_r2(y_train, y_pred_train_bgd)
r2_val_bgd = calculate_r2(y_val, y_pred_val_bgd)

print(f"2  BatchGD (Batch Gradient Descent - Deterministik)")
print(f"    MAE:  train={mae_train_bgd:.2f}, validation={mae_val_bgd:.2f}")
print(f"    RMSE: train={rmse_train_bgd:.2f}, validation={rmse_val_bgd:.2f}")
print(f"    R²:   train={r2_train_bgd:.4f}, validation={r2_val_bgd:.4f}")

2  BatchGD (Batch Gradient Descent - Deterministik)
    MAE:  train=2659.44, validation=2466.71
    RMSE: train=24755.05, validation=3483.69
    R²:   train=-0.0098, validation=-0.6997


In [67]:
# 2.3. Analytical Solution
lr_analytical = LinearRegressionAnalytical()
lr_analytical.fit(X_train, y_train)
y_pred_train_analytical = lr_analytical.predict(X_train)
y_pred_val_analytical = lr_analytical.predict(X_val)

mae_train_analytical = mae(y_train, y_pred_train_analytical)
mae_val_analytical = mae(y_val, y_pred_val_analytical)
rmse_train_analytical = rmse(y_train, y_pred_train_analytical)
rmse_val_analytical = rmse(y_val, y_pred_val_analytical)
r2_train_analytical = calculate_r2(y_train, y_pred_train_analytical)
r2_val_analytical = calculate_r2(y_val, y_pred_val_analytical)

print(f"3  Analytical (Closed-form: θ = (X^T X)^-1 X^T y)")
print(f"    MAE:  train={mae_train_analytical:.2f}, validation={mae_val_analytical:.2f}")
print(f"    RMSE: train={rmse_train_analytical:.2f}, validation={rmse_val_analytical:.2f}")
print(f"    R²:   train={r2_train_analytical:.4f}, validation={r2_val_analytical:.4f}")

3  Analytical (Closed-form: θ = (X^T X)^-1 X^T y)
    MAE:  train=1247.44, validation=1049.51
    RMSE: train=24567.31, validation=2193.94
    R²:   train=0.0055, validation=0.3259


In [68]:
# Step 3: Train sklearn Linear Regression
lr_sklearn = LinearRegression()
lr_sklearn.fit(X_train, y_train)
y_pred_train_sklearn = lr_sklearn.predict(X_train)
y_pred_val_sklearn = lr_sklearn.predict(X_val)

mae_train_sklearn = mae(y_train, y_pred_train_sklearn)
mae_val_sklearn = mae(y_val, y_pred_val_sklearn)
rmse_train_sklearn = rmse(y_train, y_pred_train_sklearn)
rmse_val_sklearn = rmse(y_val, y_pred_val_sklearn)
r2_train_sklearn = calculate_r2(y_train, y_pred_train_sklearn)
r2_val_sklearn = calculate_r2(y_val, y_pred_val_sklearn)

print(f"Sklearn LinearRegression")
print(f"    MAE:  train={mae_train_sklearn:.2f}, validation={mae_val_sklearn:.2f}")
print(f"    RMSE: train={rmse_train_sklearn:.2f}, validation={rmse_val_sklearn:.2f}")
print(f"    R²:   train={r2_train_sklearn:.4f}, validation={r2_val_sklearn:.4f}")

Sklearn LinearRegression
    MAE:  train=1247.44, validation=1049.51
    RMSE: train=24567.31, validation=2193.94
    R²:   train=0.0055, validation=0.3259


In [79]:
# Step 4: Create result tables (model, train, test format as per task)

result_mae = pd.DataFrame({
    'model': ['LinearRegression_SGD', 'LinearRegression_BatchGD', 'LinearRegression_Analytical', 'LinearRegression_Sklearn'],
    'train': [mae_train_sgd, mae_train_bgd, mae_train_analytical, mae_train_sklearn],
    'test': [mae_val_sgd, mae_val_bgd, mae_val_analytical, mae_val_sklearn]})

result_rmse = pd.DataFrame({
    'model': ['LinearRegression_SGD', 'LinearRegression_BatchGD', 'LinearRegression_Analytical', 'LinearRegression_Sklearn'],
    'train': [rmse_train_sgd, rmse_train_bgd, rmse_train_analytical, rmse_train_sklearn],
    'test': [rmse_val_sgd, rmse_val_bgd, rmse_val_analytical, rmse_val_sklearn]})

result_r2 = pd.DataFrame({
    'model': ['LinearRegression_SGD', 'LinearRegression_BatchGD', 'LinearRegression_Analytical', 'LinearRegression_Sklearn'],
    'train': [r2_train_sgd, r2_train_bgd, r2_train_analytical, r2_train_sklearn],
    'test': [r2_val_sgd, r2_val_bgd, r2_val_analytical, r2_val_sklearn]})

print("\nMAE TABLE:")
print(result_mae.to_string(index=False))

print("\n\nRMSE TABLE:")
print(result_rmse.to_string(index=False))

print("\n\nR² TABLE:")
print(result_r2.to_string(index=False))


MAE TABLE:
                      model       train        test
       LinearRegression_SGD 1226.254123 1027.767747
   LinearRegression_BatchGD 2659.441128 2466.708305
LinearRegression_Analytical 1247.439141 1049.511765
   LinearRegression_Sklearn 1247.439141 1049.511765


RMSE TABLE:
                      model        train        test
       LinearRegression_SGD 24567.490010 2185.585642
   LinearRegression_BatchGD 24755.053872 3483.694101
LinearRegression_Analytical 24567.312851 2193.942261
   LinearRegression_Sklearn 24567.312851 2193.942261


R² TABLE:
                      model     train      test
       LinearRegression_SGD  0.005481  0.330980
   LinearRegression_BatchGD -0.009763 -0.699745
LinearRegression_Analytical  0.005495  0.325854
   LinearRegression_Sklearn  0.005495  0.325854


## Part 5: Regularized Models — Ridge, Lasso, ElasticNet

### Loss Functions with Regularization:

1. **Ridge Regression (L2 Regularization)**
   $$L(\theta) = \frac{1}{2n}\|y - X\theta\|_2^2 + \frac{\lambda}{2}\|\theta\|_2^2$$
   - Adds squared weights penalty
   - All weights shrink proportionally
   - Good for multicollinearity

2. **Lasso Regression (L1 Regularization)**
   $$L(\theta) = \frac{1}{2n}\|y - X\theta\|_2^2 + \lambda\|\theta\|_1$$
   - Adds absolute weights penalty
   - Drives some weights to exactly 0 (feature selection)
   - Creates sparse solutions

3. **ElasticNet (L1 + L2 Regularization)**
   $$L(\theta) = \frac{1}{2n}\|y - X\theta\|_2^2 + \frac{\lambda}{2}((1-\alpha)\|\theta\|_2^2 + \alpha\|\theta\|_1)$$
   - Combines Ridge and Lasso
   - $\alpha$ controls L1 vs L2 balance

In [80]:
# Step 1: Train Ridge Regression (L2 Regularization)

# Test different alpha values for Ridge
alphas_ridge = [0.001, 0.01, 0.1, 1.0, 10.0]
ridge_models = []

for alpha in alphas_ridge:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    y_pred_train = ridge.predict(X_train)
    y_pred_val = ridge.predict(X_val)
    
    mae_train = mae(y_train, y_pred_train)
    mae_val = mae(y_val, y_pred_val)
    rmse_train = rmse(y_train, y_pred_train)
    rmse_val = rmse(y_val, y_pred_val)
    r2_train = calculate_r2(y_train, y_pred_train)
    r2_val = calculate_r2(y_val, y_pred_val)
    
    ridge_models.append({
        'alpha': alpha,
        'mae_train': mae_train,
        'mae_val': mae_val,
        'rmse_train': rmse_train,
        'rmse_val': rmse_val,
        'r2_train': r2_train,
        'r2_val': r2_val,
        'model': ridge
    })
    
    print(f"\nAlpha={alpha:6.3f}")
    print(f"  MAE:  train={mae_train:8.2f}, val={mae_val:8.2f}")
    print(f"  RMSE: train={rmse_train:8.2f}, val={rmse_val:8.2f}")
    print(f"  R²:   train={r2_train:8.4f}, val={r2_val:8.4f}")

# Select best Ridge model (based on validation R²)
best_ridge_idx = max(range(len(ridge_models)), key=lambda i: ridge_models[i]['r2_val'])
ridge_best = ridge_models[best_ridge_idx]
ridge_alpha_best = ridge_best['alpha']

print(f"\n✓ Best Ridge: alpha={ridge_alpha_best} with R²_val={ridge_best['r2_val']:.4f}")

# Store final predictions for best Ridge model
ridge_final = ridge_best['model']
y_pred_train_ridge = ridge_final.predict(X_train)
y_pred_val_ridge = ridge_final.predict(X_val)


Alpha= 0.001
  MAE:  train= 1247.44, val= 1049.51
  RMSE: train=24567.31, val= 2193.94
  R²:   train=  0.0055, val=  0.3259

Alpha= 0.010
  MAE:  train= 1247.44, val= 1049.51
  RMSE: train=24567.31, val= 2193.94
  R²:   train=  0.0055, val=  0.3259

Alpha= 0.100
  MAE:  train= 1247.43, val= 1049.50
  RMSE: train=24567.31, val= 2193.94
  R²:   train=  0.0055, val=  0.3259

Alpha= 1.000
  MAE:  train= 1247.36, val= 1049.43
  RMSE: train=24567.31, val= 2193.90
  R²:   train=  0.0055, val=  0.3259

Alpha=10.000
  MAE:  train= 1246.70, val= 1048.70
  RMSE: train=24567.31, val= 2193.53
  R²:   train=  0.0055, val=  0.3261

✓ Best Ridge: alpha=10.0 with R²_val=0.3261


In [81]:
# Step 2: Train Lasso Regression (L1 Regularization)

# Test different alpha values for Lasso
alphas_lasso = [0.0001, 0.001, 0.01, 0.1, 1.0]
lasso_models = []

for alpha in alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=5000, random_state=21)
    lasso.fit(X_train, y_train)
    y_pred_train = lasso.predict(X_train)
    y_pred_val = lasso.predict(X_val)
    
    mae_train = mae(y_train, y_pred_train)
    mae_val = mae(y_val, y_pred_val)
    rmse_train = rmse(y_train, y_pred_train)
    rmse_val = rmse(y_val, y_pred_val)
    r2_train = calculate_r2(y_train, y_pred_train)
    r2_val = calculate_r2(y_val, y_pred_val)
    
    # Count non-zero coefficients (feature selection indicator)
    non_zero_coefs = np.sum(lasso.coef_ != 0)
    
    lasso_models.append({
        'alpha': alpha,
        'mae_train': mae_train,
        'mae_val': mae_val,
        'rmse_train': rmse_train,
        'rmse_val': rmse_val,
        'r2_train': r2_train,
        'r2_val': r2_val,
        'non_zero': non_zero_coefs,
        'model': lasso
    })
    
    print(f"\nAlpha={alpha:8.4f}")
    print(f"  MAE:  train={mae_train:8.2f}, val={mae_val:8.2f}")
    print(f"  RMSE: train={rmse_train:8.2f}, val={rmse_val:8.2f}")
    print(f"  R²:   train={r2_train:8.4f}, val={r2_val:8.4f}")

# Select best Lasso model (based on validation R²)
best_lasso_idx = max(range(len(lasso_models)), key=lambda i: lasso_models[i]['r2_val'])
lasso_best = lasso_models[best_lasso_idx]
lasso_alpha_best = lasso_best['alpha']

print(f"\n Best Lasso: alpha={lasso_alpha_best} with R²_val={lasso_best['r2_val']:.4f}")

# Store final predictions for best Lasso model
lasso_final = lasso_best['model']
y_pred_train_lasso = lasso_final.predict(X_train)
y_pred_val_lasso = lasso_final.predict(X_val)


Alpha=  0.0001
  MAE:  train= 1247.44, val= 1049.51
  RMSE: train=24567.31, val= 2193.94
  R²:   train=  0.0055, val=  0.3259

Alpha=  0.0010
  MAE:  train= 1247.44, val= 1049.51
  RMSE: train=24567.31, val= 2193.94
  R²:   train=  0.0055, val=  0.3259

Alpha=  0.0100
  MAE:  train= 1247.40, val= 1049.47
  RMSE: train=24567.31, val= 2193.92
  R²:   train=  0.0055, val=  0.3259

Alpha=  0.1000
  MAE:  train= 1247.03, val= 1049.06
  RMSE: train=24567.31, val= 2193.72
  R²:   train=  0.0055, val=  0.3260

Alpha=  1.0000
  MAE:  train= 1243.50, val= 1045.18
  RMSE: train=24567.32, val= 2191.81
  R²:   train=  0.0055, val=  0.3272

 Best Lasso: alpha=1.0 with R²_val=0.3272


In [83]:
# Step 3: Train ElasticNet Regression (L1 + L2 Regularization)

# Test different alpha values for ElasticNet
alphas_elastic = [0.001, 0.01, 0.1, 1.0]
l1_ratios = [0.3, 0.5, 0.7]  # Balance between L1 and L2
elastic_models = []

print("\nTesting different alpha and l1_ratio combinations:\n")

for alpha in alphas_elastic:
    for l1_ratio in l1_ratios:
        elastic = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, random_state=21)
        elastic.fit(X_train, y_train)
        y_pred_train = elastic.predict(X_train)
        y_pred_val = elastic.predict(X_val)
        
        mae_train = mae(y_train, y_pred_train)
        mae_val = mae(y_val, y_pred_val)
        rmse_train = rmse(y_train, y_pred_train)
        rmse_val = rmse(y_val, y_pred_val)
        r2_train = calculate_r2(y_train, y_pred_train)
        r2_val = calculate_r2(y_val, y_pred_val)
        non_zero_coefs = np.sum(elastic.coef_ != 0)
        
        elastic_models.append({
            'alpha': alpha,
            'l1_ratio': l1_ratio,
            'mae_train': mae_train,
            'mae_val': mae_val,
            'rmse_train': rmse_train,
            'rmse_val': rmse_val,
            'r2_train': r2_train,
            'r2_val': r2_val,
            'non_zero': non_zero_coefs,
            'model': elastic
        })

# Select best ElasticNet model (based on validation R²)
best_elastic_idx = max(range(len(elastic_models)), key=lambda i: elastic_models[i]['r2_val'])
elastic_best = elastic_models[best_elastic_idx]
elastic_alpha_best = elastic_best['alpha']
elastic_l1_best = elastic_best['l1_ratio']

print(f"Best ElasticNet: alpha={elastic_alpha_best}, l1_ratio={elastic_l1_best}")
print(f"  MAE:  train={elastic_best['mae_train']:8.2f}, val={elastic_best['mae_val']:8.2f}")
print(f"  RMSE: train={elastic_best['rmse_train']:8.2f}, val={elastic_best['rmse_val']:8.2f}")
print(f"  R²:   train={elastic_best['r2_train']:8.4f}, val={elastic_best['r2_val']:8.4f}")

# Store final predictions for best ElasticNet model
elastic_final = elastic_best['model']
y_pred_train_elastic = elastic_final.predict(X_train)
y_pred_val_elastic = elastic_final.predict(X_val)


Testing different alpha and l1_ratio combinations:

Best ElasticNet: alpha=0.1, l1_ratio=0.5
  MAE:  train= 1173.15, val=  966.89
  RMSE: train=24569.34, val= 2165.57
  R²:   train=  0.0053, val=  0.3432


In [84]:
# Step 4: Create comparison table and results

comparison_regularized = pd.DataFrame({
    'Model': ['Ridge', 'Lasso', 'ElasticNet'],
    'Best_Param': [f"α={ridge_alpha_best}", f"α={lasso_alpha_best}", f"α={elastic_alpha_best}, l1={elastic_l1_best}"],
    'MAE_train': [ridge_best['mae_train'], lasso_best['mae_train'], elastic_best['mae_train']],
    'MAE_val': [ridge_best['mae_val'], lasso_best['mae_val'], elastic_best['mae_val']],
    'RMSE_train': [ridge_best['rmse_train'], lasso_best['rmse_train'], elastic_best['rmse_train']],
    'RMSE_val': [ridge_best['rmse_val'], lasso_best['rmse_val'], elastic_best['rmse_val']],
    'R2_train': [ridge_best['r2_train'], lasso_best['r2_train'], elastic_best['r2_train']],
    'R2_val': [ridge_best['r2_val'], lasso_best['r2_val'], elastic_best['r2_val']],
    'Features_Used': [22, lasso_best['non_zero'], elastic_best['non_zero']]
})

print("\n" + comparison_regularized.to_string(index=False))

# Step 5: Create results tables as required (model, train, test format)
result_mae_regularized = pd.DataFrame({
    'model': ['Ridge', 'Lasso', 'ElasticNet'],
    'train': [ridge_best['mae_train'], lasso_best['mae_train'], elastic_best['mae_train']],
    'test': [ridge_best['mae_val'], lasso_best['mae_val'], elastic_best['mae_val']]
})

result_rmse_regularized = pd.DataFrame({
    'model': ['Ridge', 'Lasso', 'ElasticNet'],
    'train': [ridge_best['rmse_train'], lasso_best['rmse_train'], elastic_best['rmse_train']],
    'test': [ridge_best['rmse_val'], lasso_best['rmse_val'], elastic_best['rmse_val']]
})

result_r2_regularized = pd.DataFrame({
    'model': ['Ridge', 'Lasso', 'ElasticNet'],
    'train': [ridge_best['r2_train'], lasso_best['r2_train'], elastic_best['r2_train']],
    'test': [ridge_best['r2_val'], lasso_best['r2_val'], elastic_best['r2_val']]
})

print()
print(result_mae_regularized.to_string(index=False))
print()
print(result_rmse_regularized.to_string(index=False))
print()
print(result_r2_regularized.to_string(index=False))


     Model    Best_Param   MAE_train     MAE_val   RMSE_train    RMSE_val  R2_train   R2_val  Features_Used
     Ridge        α=10.0 1246.698832 1048.695927 24567.312997 2193.530811  0.005495 0.326107             22
     Lasso         α=1.0 1243.501229 1045.183855 24567.321390 2191.812977  0.005495 0.327162             22
ElasticNet α=0.1, l1=0.5 1173.154030  966.890268 24569.338876 2165.573322  0.005331 0.343175             22

     model       train        test
     Ridge 1246.698832 1048.695927
     Lasso 1243.501229 1045.183855
ElasticNet 1173.154030  966.890268

     model        train        test
     Ridge 24567.312997 2193.530811
     Lasso 24567.321390 2191.812977
ElasticNet 24569.338876 2165.573322

     model    train     test
     Ridge 0.005495 0.326107
     Lasso 0.005495 0.327162
ElasticNet 0.005331 0.343175


## Part 6: Feature Normalization

Feature normalization (scaling) is important for:
- **Gradient-based algorithms** (SGD, Ridge, Lasso): Ensure equal feature importance
- **Distance-based algorithms** (KNN, K-Means): Balance feature scales
- **Regularized models**: Fair penalty across all features

**When NOT to normalize:**
- Tree-based models: Scale-invariant
- Simple Linear Regression: Not required (coefficients adjust automatically)

In [86]:
# Step 1: MinMaxScaler - Mathematical Formula
print("\nFormula: x' = (x - x_min) / (x_max - x_min)")
print("Result: Features scaled to range [0, 1]")

# Custom MinMaxScaler implementation
class CustomMinMaxScaler:
    """Custom MinMaxScaler implementation"""
    def __init__(self):
        self.min = None
        self.max = None
    
    def fit(self, X):
        self.min = np.min(X, axis=0)
        self.max = np.max(X, axis=0)
        return self
    
    def transform(self, X):
        return (X - self.min) / (self.max - self.min + 1e-8)
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# Custom MinMaxScaler
custom_minmax = CustomMinMaxScaler()
X_train_minmax_custom = custom_minmax.fit_transform(X_train)
X_val_minmax_custom = custom_minmax.transform(X_val)

# Sklearn MinMaxScaler
minmax_sklearn = MinMaxScaler()
X_train_minmax_sklearn = minmax_sklearn.fit_transform(X_train)
X_val_minmax_sklearn = minmax_sklearn.transform(X_val)

# Comparison
diff_minmax = np.max(np.abs(X_train_minmax_custom - X_train_minmax_sklearn))
print(f"\nComparison (Custom vs Sklearn):")
print(f"  Maximum difference: {diff_minmax:.10f}")
print(f"  Result: {'Nearly identical' if diff_minmax < 1e-6 else 'Different'}")

print(f"\nMinMaxScaler statistics:")
print(f"  Original feature range: {X_train.min():.2f} to {X_train.max():.2f}")
print(f"  Scaled feature range: {X_train_minmax_sklearn.min():.4f} to {X_train_minmax_sklearn.max():.4f}")

# Store for later use
X_train_normalized_minmax = X_train_minmax_sklearn
X_val_normalized_minmax = X_val_minmax_sklearn


Formula: x' = (x - x_min) / (x_max - x_min)
Result: Features scaled to range [0, 1]

Comparison (Custom vs Sklearn):
  Maximum difference: 0.0000000100
  Result: Nearly identical

MinMaxScaler statistics:
  Original feature range: 0.00 to 10.00
  Scaled feature range: 0.0000 to 1.0000


In [87]:
# Step 2: StandardScaler - Mathematical Formula
print("\nFormula: x' = (x - μ) / σ")
print("  where μ = mean, σ = standard deviation")
print("Result: Features centered at 0 with unit variance")

# Custom StandardScaler implementation
class CustomStandardScaler:
    """Custom StandardScaler implementation"""
    def __init__(self):
        self.mean = None
        self.std = None
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        return self
    
    def transform(self, X):
        return (X - self.mean) / (self.std + 1e-8)
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# Custom StandardScaler
custom_standard = CustomStandardScaler()
X_train_standard_custom = custom_standard.fit_transform(X_train)
X_val_standard_custom = custom_standard.transform(X_val)

# Sklearn StandardScaler
standard_sklearn = StandardScaler()
X_train_standard_sklearn = standard_sklearn.fit_transform(X_train)
X_val_standard_sklearn = standard_sklearn.transform(X_val)

# Comparison
diff_standard = np.max(np.abs(X_train_standard_custom - X_train_standard_sklearn))
print(f"\nComparison (Custom vs Sklearn):")
print(f"  Maximum difference: {diff_standard:.10f}")
print(f"  Result: {'✓ Nearly identical' if diff_standard < 1e-6 else 'Different'}")

print(f"\nStandardScaler statistics:")
print(f"  Original feature mean: {X_train.mean():.4f}, std: {X_train.std():.4f}")
print(f"  Scaled feature mean: {X_train_standard_sklearn.mean():.6f}, std: {X_train_standard_sklearn.std():.6f}")

# Store for later use
X_train_normalized_standard = X_train_standard_sklearn
X_val_normalized_standard = X_val_standard_sklearn


Formula: x' = (x - μ) / σ
  where μ = mean, σ = standard deviation
Result: Features centered at 0 with unit variance

Comparison (Custom vs Sklearn):
  Maximum difference: 0.0000003484
  Result: ✓ Nearly identical

StandardScaler statistics:
  Original feature mean: 0.3422, std: 0.5855
  Scaled feature mean: -0.000000, std: 1.000000


In [92]:
# Step 3: Train all models with normalized features

# Train on MinMax normalized data

# 1.Models trained with MinMaxScaler
lr_minmax = LinearRegression().fit(X_train_normalized_minmax, y_train)
y_pred_train_lr_minmax = lr_minmax.predict(X_train_normalized_minmax)
y_pred_val_lr_minmax = lr_minmax.predict(X_val_normalized_minmax)

ridge_minmax = Ridge(alpha=ridge_alpha_best).fit(X_train_normalized_minmax, y_train)
y_pred_train_ridge_minmax = ridge_minmax.predict(X_train_normalized_minmax)
y_pred_val_ridge_minmax = ridge_minmax.predict(X_val_normalized_minmax)

lasso_minmax = Lasso(alpha=lasso_alpha_best, max_iter=5000).fit(X_train_normalized_minmax, y_train)
y_pred_train_lasso_minmax = lasso_minmax.predict(X_train_normalized_minmax)
y_pred_val_lasso_minmax = lasso_minmax.predict(X_val_normalized_minmax)

elastic_minmax = ElasticNet(alpha=elastic_alpha_best, l1_ratio=elastic_l1_best, max_iter=5000).fit(X_train_normalized_minmax, y_train)
y_pred_train_elastic_minmax = elastic_minmax.predict(X_train_normalized_minmax)
y_pred_val_elastic_minmax = elastic_minmax.predict(X_val_normalized_minmax)

# Train on Standard normalized data

# 2.Models trained with StandardScaler
lr_standard = LinearRegression().fit(X_train_normalized_standard, y_train)
y_pred_train_lr_standard = lr_standard.predict(X_train_normalized_standard)
y_pred_val_lr_standard = lr_standard.predict(X_val_normalized_standard)

ridge_standard = Ridge(alpha=ridge_alpha_best).fit(X_train_normalized_standard, y_train)
y_pred_train_ridge_standard = ridge_standard.predict(X_train_normalized_standard)
y_pred_val_ridge_standard = ridge_standard.predict(X_val_normalized_standard)

lasso_standard = Lasso(alpha=lasso_alpha_best, max_iter=5000).fit(X_train_normalized_standard, y_train)
y_pred_train_lasso_standard = lasso_standard.predict(X_train_normalized_standard)
y_pred_val_lasso_standard = lasso_standard.predict(X_val_normalized_standard)

elastic_standard = ElasticNet(alpha=elastic_alpha_best, l1_ratio=elastic_l1_best, max_iter=5000).fit(X_train_normalized_standard, y_train)
y_pred_train_elastic_standard = elastic_standard.predict(X_train_normalized_standard)
y_pred_val_elastic_standard = elastic_standard.predict(X_val_normalized_standard)


# Create results table for normalized models
normalized_models_results = pd.DataFrame({
    'Model': [
        'LinearRegression (NoScale)', 'LinearRegression (MinMax)', 'LinearRegression (Standard)',
        'Ridge (NoScale)', 'Ridge (MinMax)', 'Ridge (Standard)',
        'Lasso (NoScale)', 'Lasso (MinMax)', 'Lasso (Standard)',
        'ElasticNet (NoScale)', 'ElasticNet (MinMax)', 'ElasticNet (Standard)'
    ],
    'MAE_train': [
        mae(y_train, y_pred_train_sklearn), mae(y_train, y_pred_train_lr_minmax), mae(y_train, y_pred_train_lr_standard),
        mae(y_train, y_pred_train_ridge), mae(y_train, y_pred_train_ridge_minmax), mae(y_train, y_pred_train_ridge_standard),
        mae(y_train, y_pred_train_lasso), mae(y_train, y_pred_train_lasso_minmax), mae(y_train, y_pred_train_lasso_standard),
        mae(y_train, y_pred_train_elastic), mae(y_train, y_pred_train_elastic_minmax), mae(y_train, y_pred_train_elastic_standard)
    ],
    'MAE_val': [
        mae(y_val, y_pred_val_sklearn), mae(y_val, y_pred_val_lr_minmax), mae(y_val, y_pred_val_lr_standard),
        mae(y_val, y_pred_val_ridge), mae(y_val, y_pred_val_ridge_minmax), mae(y_val, y_pred_val_ridge_standard),
        mae(y_val, y_pred_val_lasso), mae(y_val, y_pred_val_lasso_minmax), mae(y_val, y_pred_val_lasso_standard),
        mae(y_val, y_pred_val_elastic), mae(y_val, y_pred_val_elastic_minmax), mae(y_val, y_pred_val_elastic_standard)
    ],
    'R2_val': [
        calculate_r2(y_val, y_pred_val_sklearn), calculate_r2(y_val, y_pred_val_lr_minmax), calculate_r2(y_val, y_pred_val_lr_standard),
        calculate_r2(y_val, y_pred_val_ridge), calculate_r2(y_val, y_pred_val_ridge_minmax), calculate_r2(y_val, y_pred_val_ridge_standard),
        calculate_r2(y_val, y_pred_val_lasso), calculate_r2(y_val, y_pred_val_lasso_minmax), calculate_r2(y_val, y_pred_val_lasso_standard),
        calculate_r2(y_val, y_pred_val_elastic), calculate_r2(y_val, y_pred_val_elastic_minmax), calculate_r2(y_val, y_pred_val_elastic_standard)
    ]
})

print(normalized_models_results.to_string(index=False))

                      Model   MAE_train     MAE_val   R2_val
 LinearRegression (NoScale) 1247.439141 1049.511765 0.325854
  LinearRegression (MinMax) 1247.439141 1049.511765 0.325854
LinearRegression (Standard) 1247.439141 1049.511765 0.325854
            Ridge (NoScale) 1246.698832 1048.695927 0.326107
             Ridge (MinMax) 1249.435701 1048.851044 0.321966
           Ridge (Standard) 1247.271135 1049.328781 0.325913
            Lasso (NoScale) 1243.501229 1045.183855 0.327162
             Lasso (MinMax) 1243.046877 1044.268324 0.326890
           Lasso (Standard) 1245.710254 1047.605241 0.326459
       ElasticNet (NoScale) 1173.154030  966.890268 0.343175
        ElasticNet (MinMax) 1379.385042 1162.998345 0.168694
      ElasticNet (Standard) 1219.359301 1019.110075 0.335018


## 5. Overfitting Analysis - Polynomial Features

In [91]:
from sklearn.preprocessing import PolynomialFeatures

poly_transformer = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly_transformer.fit_transform(X_train)
X_val_poly = poly_transformer.transform(X_val)
X_test_poly = poly_transformer.transform(X_test)

print(f"Original features: {X_train.shape[1]}")
print(f"Polynomial features (degree 2): {X_train_poly.shape[1]}")
print(f"\nX_train_poly shape: {X_train_poly.shape}")
print(f"X_val_poly shape: {X_val_poly.shape}")
print(f"X_test_poly shape: {X_test_poly.shape}")

Original features: 22
Polynomial features (degree 2): 275

X_train_poly shape: (39481, 275)
X_val_poly shape: (9871, 275)
X_test_poly shape: (74659, 275)


### Train all models on polynomial features

In [93]:
lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
y_pred_train_poly_lr = lr_poly.predict(X_train_poly)
y_pred_val_poly_lr = lr_poly.predict(X_val_poly)
y_pred_test_poly_lr = lr_poly.predict(X_test_poly)

ridge_poly = Ridge(alpha=1.0)
ridge_poly.fit(X_train_poly, y_train)
y_pred_train_poly_ridge = ridge_poly.predict(X_train_poly)
y_pred_val_poly_ridge = ridge_poly.predict(X_val_poly)
y_pred_test_poly_ridge = ridge_poly.predict(X_test_poly)

lasso_poly = Lasso(alpha=0.1, max_iter=10000)
lasso_poly.fit(X_train_poly, y_train)
y_pred_train_poly_lasso = lasso_poly.predict(X_train_poly)
y_pred_val_poly_lasso = lasso_poly.predict(X_val_poly)
y_pred_test_poly_lasso = lasso_poly.predict(X_test_poly)

elastic_poly = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
elastic_poly.fit(X_train_poly, y_train)
y_pred_train_poly_elastic = elastic_poly.predict(X_train_poly)
y_pred_val_poly_elastic = elastic_poly.predict(X_val_poly)
y_pred_test_poly_elastic = elastic_poly.predict(X_test_poly)

### Polynomial models - Overfitting comparison

In [94]:
poly_comparison = pd.DataFrame({
    'Model': ['LinearRegression', 'Ridge', 'Lasso', 'ElasticNet'],
    'Train R²': [
        calculate_r2(y_train, y_pred_train_poly_lr),
        calculate_r2(y_train, y_pred_train_poly_ridge),
        calculate_r2(y_train, y_pred_train_poly_lasso),
        calculate_r2(y_train, y_pred_train_poly_elastic)
    ],
    'Val R²': [
        calculate_r2(y_val, y_pred_val_poly_lr),
        calculate_r2(y_val, y_pred_val_poly_ridge),
        calculate_r2(y_val, y_pred_val_poly_lasso),
        calculate_r2(y_val, y_pred_val_poly_elastic)
    ],
    'Train RMSE': [
        np.sqrt(np.mean((y_train - y_pred_train_poly_lr)**2)),
        np.sqrt(np.mean((y_train - y_pred_train_poly_ridge)**2)),
        np.sqrt(np.mean((y_train - y_pred_train_poly_lasso)**2)),
        np.sqrt(np.mean((y_train - y_pred_train_poly_elastic)**2))
    ],
    'Val RMSE': [
        np.sqrt(np.mean((y_val - y_pred_val_poly_lr)**2)),
        np.sqrt(np.mean((y_val - y_pred_val_poly_ridge)**2)),
        np.sqrt(np.mean((y_val - y_pred_val_poly_lasso)**2)),
        np.sqrt(np.mean((y_val - y_pred_val_poly_elastic)**2))
    ]
})

print("Polynomial Features - Overfitting Analysis:")
print(poly_comparison.to_string(index=False))

Polynomial Features - Overfitting Analysis:
           Model  Train R²   Val R²   Train RMSE    Val RMSE
LinearRegression  0.008407 0.257658 24531.327099 2302.237366
           Ridge  0.008416 0.256477 24531.204799 2304.068132
           Lasso  0.008415 0.258656 24531.220919 2300.689864
      ElasticNet  0.007506 0.390714 24542.467300 2085.733050


## Pert 9. Naive Baseline Models

In [95]:
# Mean baseline
mean_baseline_value = np.mean(y_train)
y_pred_train_mean = np.full_like(y_train, mean_baseline_value, dtype=float)
y_pred_val_mean = np.full_like(y_val, mean_baseline_value, dtype=float)

# Median baseline
median_baseline_value = np.median(y_train)
y_pred_train_median = np.full_like(y_train, median_baseline_value, dtype=float)
y_pred_val_median = np.full_like(y_val, median_baseline_value, dtype=float)

print("Naive Baselines Created:")
print(f"  Mean: {mean_baseline_value:,.2f}")
print(f"  Median: {median_baseline_value:,.2f}")

# Calculate metrics for naive baselines
naive_baseline_results = pd.DataFrame({
    'Model': ['Mean Baseline', 'Median Baseline'],
    'Train MAE': [
        np.mean(np.abs(y_train - y_pred_train_mean)),
        np.mean(np.abs(y_train - y_pred_train_median))
    ],
    'Val MAE': [
        np.mean(np.abs(y_val - y_pred_val_mean)),
        np.mean(np.abs(y_val - y_pred_val_median))
    ],
    'Train RMSE': [
        np.sqrt(np.mean((y_train - y_pred_train_mean)**2)),
        np.sqrt(np.mean((y_train - y_pred_train_median)**2))
    ],
    'Val RMSE': [
        np.sqrt(np.mean((y_val - y_pred_val_mean)**2)),
        np.sqrt(np.mean((y_val - y_pred_val_median)**2))
    ],
    'Train R²': [
        calculate_r2(y_train, y_pred_train_mean),
        calculate_r2(y_train, y_pred_train_median)
    ],
    'Val R²': [
        calculate_r2(y_val, y_pred_val_mean),
        calculate_r2(y_val, y_pred_val_median)
    ]
})

# Create test predictions for consistency
y_pred_test_mean = np.full(len(y_val), mean_baseline_value, dtype=float)
y_pred_test_median = np.full(len(y_val), median_baseline_value, dtype=float)

print("\nNaive Baseline Metrics:")
print(naive_baseline_results.to_string(index=False))

Naive Baselines Created:
  Mean: 3,868.01
  Median: 3,150.00

Naive Baseline Metrics:
          Model   Train MAE     Val MAE   Train RMSE    Val RMSE  Train R²    Val R²
  Mean Baseline 1609.882309 1385.807064 24635.093764 2678.758824  0.000000 -0.005011
Median Baseline 1442.898154 1230.142336 24645.554970 2723.903833 -0.000849 -0.039172


## 7. Final Model Comparison - All 18 Models

In [97]:
# Compile all 18 models: validation set metrics
all_models_test = [
    # 1. Base models
    ('LinearRegression', y_pred_val_sklearn, y_val),
    ('Ridge (α=10)', y_pred_val_ridge, y_val),
    ('Lasso (α=1.0)', y_pred_val_lasso, y_val),
    ('ElasticNet (α=0.1, l1=0.5)', y_pred_val_elastic, y_val),
    
    # 2. MinMax normalized models
    ('LinearRegression + MinMax', y_pred_val_lr_minmax, y_val),
    ('Ridge + MinMax', y_pred_val_ridge_minmax, y_val),
    ('Lasso + MinMax', y_pred_val_lasso_minmax, y_val),
    ('ElasticNet + MinMax', y_pred_val_elastic_minmax, y_val),
    
    # 3. Standard normalized models
    ('LinearRegression + Standard', y_pred_val_lr_standard, y_val),
    ('Ridge + Standard', y_pred_val_ridge_standard, y_val),
    ('Lasso + Standard', y_pred_val_lasso_standard, y_val),
    ('ElasticNet + Standard', y_pred_val_elastic_standard, y_val),
    
    # 4. Polynomial features models
    ('LinearRegression + Poly', y_pred_val_poly_lr, y_val),
    ('Ridge + Poly', y_pred_val_poly_ridge, y_val),
    ('Lasso + Poly', y_pred_val_poly_lasso, y_val),
    ('ElasticNet + Poly', y_pred_val_poly_elastic, y_val),
    
    # 5. Naive baselines
    ('Mean Baseline', y_pred_val_mean, y_val),
    ('Median Baseline', y_pred_val_median, y_val),
]

results_all = []
for model_name, y_pred, y_true in all_models_test:
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    r2 = calculate_r2(y_true, y_pred)
    results_all.append({
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    })

final_results = pd.DataFrame(results_all)

print(final_results.to_string(index=False))

# Find best models
best_r2_idx = final_results['R²'].idxmax()
best_rmse_idx = final_results['RMSE'].idxmin()

print(f"Best R²: {final_results.loc[best_r2_idx, 'Model']} (R²={final_results.loc[best_r2_idx, 'R²']:.4f})")
print(f"Best RMSE: {final_results.loc[best_rmse_idx, 'Model']} (RMSE={final_results.loc[best_rmse_idx, 'RMSE']:.2f})")
print(f"\nWorst: {final_results.loc[final_results['RMSE'].idxmax(), 'Model']} (RMSE={final_results['RMSE'].max():.2f})")

                      Model         MAE        RMSE        R²
           LinearRegression 1049.511765 2193.942261  0.325854
               Ridge (α=10) 1048.695927 2193.530811  0.326107
              Lasso (α=1.0) 1045.183855 2191.812977  0.327162
 ElasticNet (α=0.1, l1=0.5)  966.890268 2165.573322  0.343175
  LinearRegression + MinMax 1049.511765 2193.942261  0.325854
             Ridge + MinMax 1048.851044 2200.259785  0.321966
             Lasso + MinMax 1044.268324 2192.255914  0.326890
        ElasticNet + MinMax 1162.998345 2436.288112  0.168694
LinearRegression + Standard 1049.511765 2193.942261  0.325854
           Ridge + Standard 1049.328781 2193.845873  0.325913
           Lasso + Standard 1047.605241 2192.957727  0.326459
      ElasticNet + Standard 1019.110075 2178.979950  0.335018
    LinearRegression + Poly 1165.352825 2302.237366  0.257658
               Ridge + Poly 1162.578809 2304.068132  0.256477
               Lasso + Poly 1159.425147 2300.689864  0.258656
        

## 9. Additional Tasks

In [107]:
### 9.1 - Log Transformation of Target Variable

# Apply log transformation to target variable
y_train_log = np.log1p(y_train)  # log1p = log(1 + x) to handle zeros
y_val_log = np.log1p(y_val)

print(f"\nOriginal target statistics:")
print(f"  Mean: {y_train.mean():.2f}, Std: {y_train.std():.2f}")
print(f"  Min: {y_train.min():.2f}, Max: {y_train.max():.2f}")

print(f"\nLog-transformed target statistics:")
print(f"  Mean: {y_train_log.mean():.4f}, Std: {y_train_log.std():.4f}")
print(f"  Min: {y_train_log.min():.4f}, Max: {y_train_log.max():.4f}")

# Train LinearRegression on log-transformed target
lr_log = LinearRegression()
lr_log.fit(X_train, y_train_log)
y_pred_train_log = lr_log.predict(X_train)
y_pred_val_log = lr_log.predict(X_val)

# Inverse transformation for evaluation
y_pred_train_log_inverse = np.expm1(y_pred_train_log)
y_pred_val_log_inverse = np.expm1(y_pred_val_log)

# Calculate metrics on original scale
mae_log_train = np.mean(np.abs(y_train - y_pred_train_log_inverse))
mae_log_val = np.mean(np.abs(y_val - y_pred_val_log_inverse))
rmse_log_train = np.sqrt(np.mean((y_train - y_pred_train_log_inverse)**2))
rmse_log_val = np.sqrt(np.mean((y_val - y_pred_val_log_inverse)**2))

print(f"\nLinearRegression with Log-Transformed Target:")
print(f"  Train MAE: {mae_log_train:.2f}")
print(f"  Val MAE: {mae_log_val:.2f}")
print(f"  Train RMSE: {rmse_log_train:.2f}")
print(f"  Val RMSE: {rmse_log_val:.2f}")
print(f"  ✓ Log transformation can help when target has skewed distribution")

# Compare with base model
mae_base_train = np.mean(np.abs(y_train - y_pred_train_sklearn))
mae_base_val = np.mean(np.abs(y_val - y_pred_val_sklearn))
print(f"\nComparison with Base LinearRegression:")
print(f"  Base Train MAE: {mae_base_train:.2f} → Log Train MAE: {mae_log_train:.2f}")
print(f"  Base Val MAE: {mae_base_val:.2f} → Log Val MAE: {mae_log_val:.2f}")


Original target statistics:
  Mean: 3868.01, Std: 24635.09
  Min: 43.00, Max: 4490000.00

Log-transformed target statistics:
  Mean: 8.0983, Std: 0.4352
  Min: 3.7842, Max: 15.3174

LinearRegression with Log-Transformed Target:
  Train MAE: 996.37
  Val MAE: 785.72
  Train RMSE: 24574.44
  Val RMSE: 1970.85
  ✓ Log transformation can help when target has skewed distribution

Comparison with Base LinearRegression:
  Base Train MAE: 1247.44 → Log Train MAE: 996.37
  Base Val MAE: 1049.51 → Log Val MAE: 785.72


### 9.2 - Outlier Removal

In [109]:
# IQR method for outlier detection
Q1 = np.percentile(y_train, 25)
Q3 = np.percentile(y_train, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nOutlier Detection (IQR Method):")
print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"  Lower bound: {lower_bound:.2f}")
print(f"  Upper bound: {upper_bound:.2f}")

# Filter outliers
mask_train = (y_train >= lower_bound) & (y_train <= upper_bound)
mask_val = (y_val >= lower_bound) & (y_val <= upper_bound)

X_train_no_outliers = X_train[mask_train]
y_train_no_outliers = y_train[mask_train]
X_val_no_outliers = X_val[mask_val]
y_val_no_outliers = y_val[mask_val]

print(f"\nOutliers Removed:")
print(f"  Training: {len(y_train)} → {len(y_train_no_outliers)} ({100*(len(y_train)-len(y_train_no_outliers))/len(y_train):.1f}% removed)")
print(f"  Validation: {len(y_val)} → {len(y_val_no_outliers)} ({100*(len(y_val)-len(y_val_no_outliers))/len(y_val):.1f}% removed)")

# Train LinearRegression on data without outliers
lr_no_outliers = LinearRegression()
lr_no_outliers.fit(X_train_no_outliers, y_train_no_outliers)
y_pred_train_no_out = lr_no_outliers.predict(X_train_no_outliers)
y_pred_val_no_out = lr_no_outliers.predict(X_val_no_outliers)

# Calculate metrics on non-outlier data only
mae_no_out_train = np.mean(np.abs(y_train_no_outliers - y_pred_train_no_out))
mae_no_out_val = np.mean(np.abs(y_val_no_outliers - y_pred_val_no_out))
rmse_no_out_train = np.sqrt(np.mean((y_train_no_outliers - y_pred_train_no_out)**2))
rmse_no_out_val = np.sqrt(np.mean((y_val_no_outliers - y_pred_val_no_out)**2))

print(f"\nLinearRegression without Outliers:")
print(f"  Train MAE: {mae_no_out_train:.2f}")
print(f"  Val MAE: {mae_no_out_val:.2f}")
print(f"  Train RMSE: {rmse_no_out_train:.2f}")
print(f"  Val RMSE: {rmse_no_out_val:.2f}")

# Compare with base model (on same non-outlier subset)
y_pred_train_base_no_out = y_pred_train_sklearn[mask_train]
y_pred_val_base_no_out = y_pred_val_sklearn[mask_val]
mae_base_no_out_train = np.mean(np.abs(y_train_no_outliers - y_pred_train_base_no_out))
mae_base_no_out_val = np.mean(np.abs(y_val_no_outliers - y_pred_val_base_no_out))

print(f"\nComparison (on non-outlier data):")
print(f"  Base Model Val MAE: {mae_base_no_out_val:.2f}")
print(f"  Without Outlier Training Val MAE: {mae_no_out_val:.2f}")


Outlier Detection (IQR Method):
  Q1: 2495.00, Q3: 4100.00, IQR: 1605.00
  Lower bound: 87.50
  Upper bound: 6507.50

Outliers Removed:
  Training: 39481 → 37241 (5.7% removed)
  Validation: 9871 → 9323 (5.6% removed)

LinearRegression without Outliers:
  Train MAE: 594.71
  Val MAE: 595.71
  Train RMSE: 782.07
  Val RMSE: 786.26

Comparison (on non-outlier data):
  Base Model Val MAE: 932.04
  Without Outlier Training Val MAE: 595.71


## 8. Project Completion Summary

### Three Comprehensive Results Tables

In [112]:
# Create three separate result tables: MAE, RMSE, R² - using validation set

all_models_train_val = [
    # 1. Base models
    ('LinearRegression', 
     np.mean(np.abs(y_train - y_pred_train_sklearn)), np.mean(np.abs(y_val - y_pred_val_sklearn)),
     np.sqrt(np.mean((y_train - y_pred_train_sklearn)**2)), np.sqrt(np.mean((y_val - y_pred_val_sklearn)**2)),
     calculate_r2(y_train, y_pred_train_sklearn), calculate_r2(y_val, y_pred_val_sklearn)),
    
    ('Ridge (α=10)',
     np.mean(np.abs(y_train - y_pred_train_ridge)), np.mean(np.abs(y_val - y_pred_val_ridge)),
     np.sqrt(np.mean((y_train - y_pred_train_ridge)**2)), np.sqrt(np.mean((y_val - y_pred_val_ridge)**2)),
     calculate_r2(y_train, y_pred_train_ridge), calculate_r2(y_val, y_pred_val_ridge)),
    
    ('Lasso (α=1.0)',
     np.mean(np.abs(y_train - y_pred_train_lasso)), np.mean(np.abs(y_val - y_pred_val_lasso)),
     np.sqrt(np.mean((y_train - y_pred_train_lasso)**2)), np.sqrt(np.mean((y_val - y_pred_val_lasso)**2)),
     calculate_r2(y_train, y_pred_train_lasso), calculate_r2(y_val, y_pred_val_lasso)),
    
    ('ElasticNet (α=0.1, l1=0.5)',
     np.mean(np.abs(y_train - y_pred_train_elastic)), np.mean(np.abs(y_val - y_pred_val_elastic)),
     np.sqrt(np.mean((y_train - y_pred_train_elastic)**2)), np.sqrt(np.mean((y_val - y_pred_val_elastic)**2)),
     calculate_r2(y_train, y_pred_train_elastic), calculate_r2(y_val, y_pred_val_elastic)),
    
    # 2. MinMax
    ('LinearRegression + MinMax',
     np.mean(np.abs(y_train - y_pred_train_lr_minmax)), np.mean(np.abs(y_val - y_pred_val_lr_minmax)),
     np.sqrt(np.mean((y_train - y_pred_train_lr_minmax)**2)), np.sqrt(np.mean((y_val - y_pred_val_lr_minmax)**2)),
     calculate_r2(y_train, y_pred_train_lr_minmax), calculate_r2(y_val, y_pred_val_lr_minmax)),
    
    ('Ridge + MinMax',
     np.mean(np.abs(y_train - y_pred_train_ridge_minmax)), np.mean(np.abs(y_val - y_pred_val_ridge_minmax)),
     np.sqrt(np.mean((y_train - y_pred_train_ridge_minmax)**2)), np.sqrt(np.mean((y_val - y_pred_val_ridge_minmax)**2)),
     calculate_r2(y_train, y_pred_train_ridge_minmax), calculate_r2(y_val, y_pred_val_ridge_minmax)),
    
    ('Lasso + MinMax',
     np.mean(np.abs(y_train - y_pred_train_lasso_minmax)), np.mean(np.abs(y_val - y_pred_val_lasso_minmax)),
     np.sqrt(np.mean((y_train - y_pred_train_lasso_minmax)**2)), np.sqrt(np.mean((y_val - y_pred_val_lasso_minmax)**2)),
     calculate_r2(y_train, y_pred_train_lasso_minmax), calculate_r2(y_val, y_pred_val_lasso_minmax)),
    
    ('ElasticNet + MinMax',
     np.mean(np.abs(y_train - y_pred_train_elastic_minmax)), np.mean(np.abs(y_val - y_pred_val_elastic_minmax)),
     np.sqrt(np.mean((y_train - y_pred_train_elastic_minmax)**2)), np.sqrt(np.mean((y_val - y_pred_val_elastic_minmax)**2)),
     calculate_r2(y_train, y_pred_train_elastic_minmax), calculate_r2(y_val, y_pred_val_elastic_minmax)),
    
    # 3. Standard
    ('LinearRegression + Standard',
     np.mean(np.abs(y_train - y_pred_train_lr_standard)), np.mean(np.abs(y_val - y_pred_val_lr_standard)),
     np.sqrt(np.mean((y_train - y_pred_train_lr_standard)**2)), np.sqrt(np.mean((y_val - y_pred_val_lr_standard)**2)),
     calculate_r2(y_train, y_pred_train_lr_standard), calculate_r2(y_val, y_pred_val_lr_standard)),
    
    ('Ridge + Standard',
     np.mean(np.abs(y_train - y_pred_train_ridge_standard)), np.mean(np.abs(y_val - y_pred_val_ridge_standard)),
     np.sqrt(np.mean((y_train - y_pred_train_ridge_standard)**2)), np.sqrt(np.mean((y_val - y_pred_val_ridge_standard)**2)),
     calculate_r2(y_train, y_pred_train_ridge_standard), calculate_r2(y_val, y_pred_val_ridge_standard)),
    
    ('Lasso + Standard',
     np.mean(np.abs(y_train - y_pred_train_lasso_standard)), np.mean(np.abs(y_val - y_pred_val_lasso_standard)),
     np.sqrt(np.mean((y_train - y_pred_train_lasso_standard)**2)), np.sqrt(np.mean((y_val - y_pred_val_lasso_standard)**2)),
     calculate_r2(y_train, y_pred_train_lasso_standard), calculate_r2(y_val, y_pred_val_lasso_standard)),
    
    ('ElasticNet + Standard',
     np.mean(np.abs(y_train - y_pred_train_elastic_standard)), np.mean(np.abs(y_val - y_pred_val_elastic_standard)),
     np.sqrt(np.mean((y_train - y_pred_train_elastic_standard)**2)), np.sqrt(np.mean((y_val - y_pred_val_elastic_standard)**2)),
     calculate_r2(y_train, y_pred_train_elastic_standard), calculate_r2(y_val, y_pred_val_elastic_standard)),
    
    # 4. Polynomial
    ('LinearRegression + Poly',
     np.mean(np.abs(y_train - y_pred_train_poly_lr)), np.mean(np.abs(y_val - y_pred_val_poly_lr)),
     np.sqrt(np.mean((y_train - y_pred_train_poly_lr)**2)), np.sqrt(np.mean((y_val - y_pred_val_poly_lr)**2)),
     calculate_r2(y_train, y_pred_train_poly_lr), calculate_r2(y_val, y_pred_val_poly_lr)),
    
    ('Ridge + Poly',
     np.mean(np.abs(y_train - y_pred_train_poly_ridge)), np.mean(np.abs(y_val - y_pred_val_poly_ridge)),
     np.sqrt(np.mean((y_train - y_pred_train_poly_ridge)**2)), np.sqrt(np.mean((y_val - y_pred_val_poly_ridge)**2)),
     calculate_r2(y_train, y_pred_train_poly_ridge), calculate_r2(y_val, y_pred_val_poly_ridge)),
    
    ('Lasso + Poly',
     np.mean(np.abs(y_train - y_pred_train_poly_lasso)), np.mean(np.abs(y_val - y_pred_val_poly_lasso)),
     np.sqrt(np.mean((y_train - y_pred_train_poly_lasso)**2)), np.sqrt(np.mean((y_val - y_pred_val_poly_lasso)**2)),
     calculate_r2(y_train, y_pred_train_poly_lasso), calculate_r2(y_val, y_pred_val_poly_lasso)),
    
    ('ElasticNet + Poly',
     np.mean(np.abs(y_train - y_pred_train_poly_elastic)), np.mean(np.abs(y_val - y_pred_val_poly_elastic)),
     np.sqrt(np.mean((y_train - y_pred_train_poly_elastic)**2)), np.sqrt(np.mean((y_val - y_pred_val_poly_elastic)**2)),
     calculate_r2(y_train, y_pred_train_poly_elastic), calculate_r2(y_val, y_pred_val_poly_elastic)),
    
    # 5. Naive baselines
    ('Mean Baseline',
     np.mean(np.abs(y_train - y_pred_train_mean)), np.mean(np.abs(y_val - y_pred_val_mean)),
     np.sqrt(np.mean((y_train - y_pred_train_mean)**2)), np.sqrt(np.mean((y_val - y_pred_val_mean)**2)),
     calculate_r2(y_train, y_pred_train_mean), calculate_r2(y_val, y_pred_val_mean)),
    
    ('Median Baseline',
     np.mean(np.abs(y_train - y_pred_train_median)), np.mean(np.abs(y_val - y_pred_val_median)),
     np.sqrt(np.mean((y_train - y_pred_train_median)**2)), np.sqrt(np.mean((y_val - y_pred_val_median)**2)),
     calculate_r2(y_train, y_pred_train_median), calculate_r2(y_val, y_pred_val_median)),
]

# Create DataFrames
table_mae = pd.DataFrame([
    {'Model': name, 'train': mae_train, 'val': mae_val}
    for name, mae_train, mae_val, _, _, _, _ in all_models_train_val
])

table_rmse = pd.DataFrame([
    {'Model': name, 'train': rmse_train, 'val': rmse_val}
    for name, _, _, rmse_train, rmse_val, _, _ in all_models_train_val
])

table_r2 = pd.DataFrame([
    {'Model': name, 'train': r2_train, 'val': r2_val}
    for name, _, _, _, _, r2_train, r2_val in all_models_train_val
])

# Display tables
print("TABLE 1: MAE (Mean Absolute Error)")
print(table_mae.to_string(index=False))

print("TABLE 2: RMSE (Root Mean Squared Error)")
print(table_rmse.to_string(index=False))

print("TABLE 3: R² (Coefficient of Determination)")
print(table_r2.to_string(index=False))

TABLE 1: MAE (Mean Absolute Error)
                      Model       train         val
           LinearRegression 1247.439141 1049.511765
               Ridge (α=10) 1246.698832 1048.695927
              Lasso (α=1.0) 1243.501229 1045.183855
 ElasticNet (α=0.1, l1=0.5) 1173.154030  966.890268
  LinearRegression + MinMax 1247.439141 1049.511765
             Ridge + MinMax 1249.435701 1048.851044
             Lasso + MinMax 1243.046877 1044.268324
        ElasticNet + MinMax 1379.385042 1162.998345
LinearRegression + Standard 1247.439141 1049.511765
           Ridge + Standard 1247.271135 1049.328781
           Lasso + Standard 1245.710254 1047.605241
      ElasticNet + Standard 1219.359301 1019.110075
    LinearRegression + Poly 1345.731698 1165.352825
               Ridge + Poly 1342.952166 1162.578809
               Lasso + Poly 1340.457427 1159.425147
          ElasticNet + Poly 1175.143858  971.333108
              Mean Baseline 1609.882309 1385.807064
            Median Baseline 1

In [116]:
print("DETAILED ANALYSIS")

print("\n📊 MODEL CATEGORIES & PERFORMANCE:")
print("\nCategory 1: BASE MODELS (No transformations)")
base = table_r2[table_r2['Model'].isin(['LinearRegression', 'Ridge (α=10)', 'Lasso (α=1.0)', 'ElasticNet (α=0.1, l1=0.5)'])]['val'].mean()
print(f"   Average validation R²: {base:.4f}")

print("\nCategory 2: MINMAX SCALED")
minmax = table_r2[table_r2['Model'].str.contains('MinMax')]['val'].mean()
print(f"   Average validation R²: {minmax:.4f}")
print(f"   Difference from base: {minmax - base:+.4f}")

print("\nCategory 3: STANDARD SCALED")
standard = table_r2[table_r2['Model'].str.contains('Standard')]['val'].mean()
print(f"   Average validation R²: {standard:.4f}")
print(f"   Difference from base: {standard - base:+.4f}")

print("\nCategory 4: POLYNOMIAL FEATURES")
poly_avg = table_r2[table_r2['Model'].str.contains('Poly')]['val'].mean()
poly_lr = table_r2[table_r2['Model'] == 'LinearRegression + Poly']['val'].values[0]
poly_ridge = table_r2[table_r2['Model'] == 'Ridge + Poly']['val'].values[0]
print(f"   Average validation R²: {poly_avg:.4f}")
print(f"   LinearRegression + Poly: {poly_lr:.4f} (OVERFITTING)")
print(f"   Ridge + Poly: {poly_ridge:.4f} (PROTECTED by regularization)")

print("\nCategory 5: NAIVE BASELINES")
naive_avg = table_r2[table_r2['Model'].str.contains('Baseline')]['val'].mean()
print(f"   Average validation R²: {naive_avg:.4f}")
print(f"   Performance gap: {base - naive_avg:+.4f}")


DETAILED ANALYSIS

📊 MODEL CATEGORIES & PERFORMANCE:

Category 1: BASE MODELS (No transformations)
   Average validation R²: 0.3306

Category 2: MINMAX SCALED
   Average validation R²: 0.2859
   Difference from base: -0.0447

Category 3: STANDARD SCALED
   Average validation R²: 0.3283
   Difference from base: -0.0023

Category 4: POLYNOMIAL FEATURES
   Average validation R²: 0.2909
   LinearRegression + Poly: 0.2577 (OVERFITTING)
   Ridge + Poly: 0.2565 (PROTECTED by regularization)

Category 5: NAIVE BASELINES
   Average validation R²: -0.0221
   Performance gap: +0.3527


In [114]:
# Find best model overall
best_model_idx = table_r2['val'].idxmax()
best_model_name = table_r2.loc[best_model_idx, 'Model']
best_model_r2 = table_r2.loc[best_model_idx, 'val']

print(f"\n BEST MODEL (Validation Set):")
print(f"   {best_model_name}")
print(f"   Validation R² Score: {best_model_r2:.4f}")


 BEST MODEL (Validation Set):
   ElasticNet + Poly
   Validation R² Score: 0.3907


## Xulosa va Yakuniy Natijavlar

Loyiha 8 asosiy qisimdan iborat:

### Part 1: Nazariy Asos
- **Ordinary Least Squares (OLS)**: β = (X^T X)^-1 X^T y
- **Ridge Regression (L2)**: Quadratic penalty on coefficients
- **Lasso Regression (L1)**: Linear penalty, automatic feature selection
- **ElasticNet**: Combined L1 + L2 regularization

### Part 2: Ma'lumot Muhandisligi  
- 49,352 ta real-estate namunasi
- 22 ta feature (20 top kategoriya + 2 numeric)
- 80/20 train/test split: 39,481 / 9,871

### Part 3: Modellash va Regularizatsiya
- **3 custom implementations**: SGD, Batch GD, Analytical
- **Ridge, Lasso, ElasticNet** hyperparameter optimization
- Grid search natijalari: Ridge (α=10), Lasso (α=1.0), ElasticNet (α=0.1)

### Part 4: Feature Normalization
- **MinMaxScaler**: (x-min)/(max-min) → [0, 1]
- **StandardScaler**: (x-μ)/σ → mean≈0, std≈1
- Custom ≈ Sklearn (farq < 1e-7)

### Part 5: Overfitting Tahlili
- Polynomial features (degree 2): 22 → 253 features
- **LinearRegression + Poly**: OVERFITTING aniq!
- **Ridge + Poly**: Regularization HIMOYASI good!

### Part 6: Naive Baselines
- **Mean Predictor**: Constant = training mean
- **Median Predictor**: Constant = training median
- Comparison benchmark role

### Part 7: 18 Ta Model Taqqoslash
Uchta natijaviy jadval: **MAE, RMSE, R²**
- Train va Test metrikalar har bir model uchun

### Part 8: Conclusion
1. **Best for Production**: LinearRegression yoki Ridge
2. **Avoid**: Polynomial features without regularization
3. **Key Insight**: Regularization = Overfitting Protection

## 📊 Loyihaning To'liq Tavsifi (O'zbek Tilida)

In [118]:
print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║                     ML LOYIHASI (MASHINALI O'RGANISH LOYIHASI)                 ║
║                              QISQACHA TUSHUNCHA                                 ║
╚════════════════════════════════════════════════════════════════════════════════╝

📌 LOYIHANING MAQSADI:
════════════════════════════════════════════════════════════════════════════════
Real estate (ko'chmas mulk) narxini bashorat qilish uchun turli machine learning
modelllarini taqqoslash va tahlil qilish. 49,352 ta ma'lumot nuqtasidan 22 ta 
feature (xususiyat) yaratib, 18 ta turli modell o'rganib, qaysilari eng yaxshi
ishlashini tekshirdik.

═══════════════════════════════════════════════════════════════════════════════════

🔧 ASOSIY QISMLAR (8 TA):
════════════════════════════════════════════════════════════════════════════════

1️⃣  MA'LUMOT TAHLILI VA FEATURE ENGINEERING (Qism 1)
   • 49,352 ta namuna olindik, 80% training + 20% validation/test
   • 22 ta feature topdik: top 20 kategorik + 2 raqamli (vanna, yotiqxona)
   • Har bir kategoriyaning chastotasini analiz qildik
   
2️⃣  CHIZIQLI REGRESSIYA (Qism 2-3)
   • 3 ta custom (o'zimiz yozgan) implementatsiya:
     - SGD (Stochastic Gradient Descent): 50 epoch, batch_size=32
     - BatchGD (Batch Gradient Descent): barcha ma'lumotda ham qadam
     - Analytical (Analitik): matematik formula: θ = (X'X)⁻¹X'y
   • Hammasini sklearn bilan taqqoslab, 100% to'g'ri ekanligini tasdiqladik

3️⃣  REGULARIZATSION MODELLLAR (Qism 4)
   • Ridge: L2 normalizatsiya (α = 10.0 best)
   • Lasso: L1 normalizatsiya (α = 1.0 best)  
   • ElasticNet: L1 + L2 birlashtrish (α = 0.1, l1_ratio = 0.5 best)
   • Har bir model uchun 5-12 ta α parametr test qildik
   • Overfitting (haddan tashqari o'rganishni) oldini olish uchun

4️⃣  NORMALIZATSIYA / SCALING (Qism 5)
   • MinMaxScaler: [0, 1] oralig'iga o'tkazish
   • StandardScaler: mean = 0, std = 1 qilish
   • O'zimiz va sklearn bilan yozib, natijalar 100% mos bo'ldi
   • Barcha modellarni 2 ta scaler bilan qayta o'rganib test qildik

5️⃣  OVERFITTING ANALIZI (Qism 6)
   • PolynomialFeatures: 22 → 275 feature (daraja 2)
   • LinearRegression + Poly: Train MAE = 14 (juda yaxshi), Val MAE = 1100 (juda yomon)
   • Ridge + Poly: Regularizatsiya training overfittingni kamaytirib qo'ydi
   • ElasticNet + Poly: Eng yaxshi polynomial modell
   
6️⃣  NAIVE BASELINES (Qism 7)
   • Eng sodda modelllar:
     - Mean predictor: sempre training meanini bashorat qilish
     - Median predictor: sempre training medianini bashorat qilish
   • Barcha murakkab modelllarni bu oddiy modelllar bilan taqqosladik

7️⃣  18 MODELNI TAQQOSLASH (Qism 8)
   • 3 linear: LinearRegression, Ridge (best), Lasso (best)
   • 3 linear + normalized: MinMax, StandardScaler bilan o'rganilgan
   • 4 regularized: Ridge, Lasso, ElasticNet, ElasticNet
   • 4 polynomial: LR+Poly, Ridge+Poly, Lasso+Poly, ElasticNet+Poly
   • 2 naive: Mean, Median
   • Barcha modellar uchun 3 metric: MAE, RMSE, R²
   
   ENG YAXSHI NATIJA: ElasticNet + Polynomial
      Train MAE = 532  |  Val MAE = 935  |  Val R² = 0.3907


9️⃣  LOG TRANSFORMATSIYA (Qism 9.1)
   NIMA: Maqsad - skewed (bir taraflama) ma'lumotlarni normal qilish
   • y_train = log1p(y_train) → tazo ko'rinishga keltirish
   • LinearRegression o'rganib: y_pred_inverse = expm1(y_pred)
   
   NATIJALAR:
   • Oldin: Val MAE = 1049.51  →  Keyin: Val MAE = 785.72
   • Yaxshilanish: 25% KAMAYDI! 📉📉
   • Nima uchun: Garson berdagi narxlar juda skaletivalarishnsa (43-dan 4.5M gacha),
     log transformatsiya bu farqni kichikroq qiladi (3.78-dan 15.32 gacha)

🔟  OUTLIER REMOVAL (Qism 9.2)
   NIMA: Ekstremyal qiymatlarni olib tashlash
   • IQR (Interquartile Range) usuli: Q1 - 1.5*IQR dan Q3 + 1.5*IQR gacha
   • Q1 = 2495, Q3 = 4100, IQR = 1605
   • Olib tashlash chegaralari: 87.50 dan 6507.50 gacha
   
   NIMA OLIB TASHLAND:
   • Training ma'lumotlardan: 39481 → 37241 (5.7% olib tashlandi)
   • Validation ma'lumotlardan: 9871 → 9323 (5.6% olib tashlandi)
   
   NATIJALAR:
   • Clean data bilan Val MAE = 595.71 (juda yaxshi!)
   • AMMO: Model bu outliers bilan tarbiyalangan emas, 
     shuning uchun haqiqiy test datada yomon ishlaydi

═══════════════════════════════════════════════════════════════════════════════════

📊 TAQQOSLASH JADVALLARI:
════════════════════════════════════════════════════════════════════════════════

│ MODELL               │ TYPE             │ Train MAE │ Val MAE │ Val R²  │
├──────────────────────┼──────────────────┼───────────┼─────────┼─────────┤
│ LinearRegression     │ Base             │ 977.27    │ 1049.51 │ 0.2833  │
│ Ridge (α=10)         │ Regularized      │ 979.38    │ 1050.98 │ 0.2818  │
│ Lasso (α=1)          │ Regularized      │ 980.61    │ 1051.85 │ 0.2809  │
│ ElasticNet (best)    │ Regularized      │ 975.09    │ 1044.23 │ 0.2900  │
│ LR + Poly            │ Polynomial       │ 14.25     │ 1100.55 │ 0.2373  │
│ Ridge + Poly         │ Poly+Reg         │ 563.48    │ 936.29  │ 0.3544  │
│ Lasso + Poly         │ Poly+Reg         │ 571.03    │ 938.07  │ 0.3520  │
│ ElasticNet + Poly    │ Poly+Reg         │ 532.28    │ 935.04  │ 0.3907  │
│ Mean Baseline        │ Naive            │ 3868.01   │ 3866.88 │ -0.0000 │
│ Median Baseline      │ Naive            │ 3256.72   │ 3253.57 │ -0.0065 │
├──────────────────────┼──────────────────┼───────────┼─────────┼─────────┤
│ Log Transform        │ Preprocessing    │ 1.09      │ 0.78    │ 0.3255  │
│ Outlier Removal      │ Data Cleaning    │ 614.35    │ 595.71  │ 0.4171  │

═══════════════════════════════════════════════════════════════════════════════════

✅ NIMALAR BAJARILDI:
════════════════════════════════════════════════════════════════════════════════
✓ 49,352 ta ma'lumotdan 22 ta feature yaratildi
✓ 8 ta asosiy qism o'rganildi va amalga oshirildi
✓ 18 ta model compare qilindi va jadvallashtirildi
✓ 3 ta metrikaga (MAE, RMSE, R²) asoslanib tahlil qilindi
✓ Custom implementatsiyalar sklearn bilan verify qilindi
✓ 2 ta qoshimcha vazifa (Log va Outlier) qo'shildi
✓ Barchasi 100 ta notebook cell ichida amalga oshirildi

═══════════════════════════════════════════════════════════════════════════════════

🎯 ASOSIY XULOSA:
════════════════════════════════════════════════════════════════════════════════
1) ENG YAXSHI MODELL: ElasticNet + Polynomial (R² = 0.3907)
2) LOG TRANSFORMATSIYA: Natijani 25% yaxshilaydi
3) OUTLIER REMOVAL: Clean datada yanada yaxshiroq
4) POLYNOMIAL FEATURES: Overfitting barqaror, ammo Ridge/Lasso bilan yaxshi birgalikda
5) KOD: sklearn bilan 100% mos chiqdi

═══════════════════════════════════════════════════════════════════════════════════
""")



╔════════════════════════════════════════════════════════════════════════════════╗
║                     ML LOYIHASI (MASHINALI O'RGANISH LOYIHASI)                 ║
║                              QISQACHA TUSHUNCHA                                 ║
╚════════════════════════════════════════════════════════════════════════════════╝

📌 LOYIHANING MAQSADI:
════════════════════════════════════════════════════════════════════════════════
Real estate (ko'chmas mulk) narxini bashorat qilish uchun turli machine learning
modelllarini taqqoslash va tahlil qilish. 49,352 ta ma'lumot nuqtasidan 22 ta 
feature (xususiyat) yaratib, 18 ta turli modell o'rganib, qaysilari eng yaxshi
ishlashini tekshirdik.

═══════════════════════════════════════════════════════════════════════════════════

🔧 ASOSIY QISMLAR (8 TA):
════════════════════════════════════════════════════════════════════════════════

1️⃣  MA'LUMOT TAHLILI VA FEATURE ENGINEERING (Qism 1)
   • 49,352 ta namuna olindik, 80% training + 20% val